# Lab 14-06 · Pipelining and Hazards

Companion to [chapter 14-06](../../chapters/14-06-pipelining-and-hazards.md) and the pipelined Verilog core in this directory. Each micro-program is assembled with LLVM, run on the Python simulator to get the hazard model's prediction ($N + 4 + S + 2T$), and, if Icarus Verilog is installed, run on the core, whose counts and register writes are compared with the prediction. Run the cells in order.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../verify').resolve()))
import _pipe
HDR = '    .text\n    .globl _start\n'
def show(name, asm, array=None):
    r = _pipe.run(HDR + asm, array)
    line = f"{name:28s} N = {r['instr']:3d}, S = {r['stalls']}, T = {r['taken']}, model {r['model_cycles']:4d} cycles"
    if 'cycles' in r:
        line += f" | Verilog {r['cycles']:4d} cycles, {r['v_stalls']} stalls, {r['v_flushes']} flushes, trace {'matches' if r['trace_match'] else 'DIFFERS'}"
    print(line)
    return r
print('Icarus Verilog available:', _pipe.have_iverilog())

Icarus Verilog available: True


## Part 1 · Data hazards

**Predict first:** how many stalls does each sequence cause?

In [2]:
show('forwarding only (Ex. 3)', '''_start:
    li x6, 3
    li x7, 4
    li x9, 1
    add x5, x6, x7
    sub x8, x5, x9
    or x10, x5, x8
    ebreak
''')
show('load-use (Ex. 4a)', '''_start:
    li x10, 0x2000
    li x7, 5
    lw x5, 0(x10)
    add x6, x5, x7
    ebreak
''', [42])
show('scheduled (Ex. 4b)', '''_start:
    li x10, 0x2000
    lw x5, 0(x10)
    li x7, 5
    add x6, x5, x7
    ebreak
''', [42])
r = show('forwarding priority (P. 5)', '''_start:
    li x5, 3
    li x5, 10
    add x6, x5, x5
    ebreak
''')
print('x6 =', r['sim'].x[6])

forwarding only (Ex. 3)      N =   7, S = 0, T = 0, model   11 cycles | Verilog   11 cycles, 0 stalls, 0 flushes, trace matches
load-use (Ex. 4a)            N =   5, S = 1, T = 0, model   10 cycles | Verilog   10 cycles, 1 stalls, 0 flushes, trace matches


scheduled (Ex. 4b)           N =   5, S = 0, T = 0, model    9 cycles | Verilog    9 cycles, 0 stalls, 0 flushes, trace matches


forwarding priority (P. 5)   N =   4, S = 0, T = 0, model    8 cycles | Verilog    8 cycles, 0 stalls, 0 flushes, trace matches
x6 = 20


## Part 2 · Control hazards and loops

Each taken branch or jump costs two flush cycles. Measure the cost per element of the three sum loops of Example 14-06.5 by running 10 and 11 elements.

In [3]:
lab02 = Path('../14-02-lab')
SCHED = '''sum:
    addi t0, zero, 0
    beq  a1, zero, 2f
1:  lw   t1, 0(a0)
    addi a0, a0, 4
    addi a1, a1, -1
    add  t0, t0, t1
    bne  a1, zero, 1b
2:  addi a0, t0, 0
    jalr zero, 0(ra)
'''
HEAD = '_start:\n    li sp, 0x4000\n    li a0, 0x2000\n    li a1, {n}\n    call sum\n    ebreak\n'
for name, body in (('top-tested', (lab02 / 'sum_rv32.s').read_text()), ('rotated', (lab02 / 'sum_rv32_rot.s').read_text()), ('scheduled', SCHED)):
    c = [_pipe.run(HDR + HEAD.format(n=n) + body, list(range(1, 13))) for n in (10, 11)]
    key = 'cycles' if 'cycles' in c[0] else 'model_cycles'
    print(f'{name:11s}: {c[1][key] - c[0][key]} cycles per element')

top-tested : 9 cycles per element


rotated    : 8 cycles per element


scheduled  : 7 cycles per element


## Part 3 · The whole program

The lab program's prediction and, if available, the Verilog result.

In [4]:
r = _pipe.run(Path('program.s').read_text(), [7 * i - 20 for i in range(10)])
print(f"N = {r['instr']}, S = {r['stalls']}, T = {r['taken']}: model {r['model_cycles']} cycles, CPI = {r['model_cycles'] / r['instr']:.3f}")
if 'report' in r:
    print(r['report'].strip())

N = 1924, S = 10, T = 456: model 2850 cycles, CPI = 1.481
sum = 115, fib(10) = 55, 2850 cycles for 1924 instructions (CPI = 1.481), 10 load-use stalls, 456 flushes, 0.812 us at 285 ps
TEST PASSED
rv32_pipe_tb.v:69: $finish called at 855150 (1ps)


**Explore.** Write a loop that sums an array *and* counts its negative elements. Predict its stalls and flushes per element before running it, then schedule it to remove the stalls.